# AI Data-Quality Triage — a working demo

**The claim this notebook demonstrates:** *AI agents can take a raw data-quality alert and turn it into a complete, trustworthy incident diagnosis — automatically, in seconds, with humans involved only where risk demands it.*

The story: a small online shop (mugs, t-shirts, hoodies…). Something breaks in its sales data. Watch a team of small AI agents figure out **what** broke, **why**, **how much it matters**, and **what to do about it**.

What each part proves:

1. **Detection can stay dumb** — a simple threshold check fires the alert; a chart shows you the anomaly before any AI runs.
2. **The agents genuinely reason** — Act 2 reruns the *identical* agents on a different failure and gets a different, correct diagnosis.
3. **AI output can be trusted enough to act on** — Pydantic validates structure *and* meaning ("a confident root cause must cite evidence").
4. **Risk is governable** — P1 / high-risk / low-confidence outcomes route to a human; the rest may auto-apply.
5. **Waste is preventable** — a dedup gate short-circuits repeat alerts before any agent spends money.

| Layer | In this notebook | In production |
|---|---|---|
| Warehouse | in-memory SQLite | your real warehouse |
| Lineage + catalog | Python dicts | Glue / Collibra / dbt / OpenLineage via MCP |
| Detection | real checks (run here) | Great Expectations + ML drift |
| **AI agents** | **real** | **real (same code)** |
| **Pydantic validation** | **real** | **real** |
| **Orchestrator** | **real** | **real** |

> Runs out of the box in **MOCK mode** — no API key needed. Add an OpenAI or Anthropic key to a `.env` file to watch the agents reason live.
>
> *Honest boundary:* the enterprise plumbing is stubbed, so this demo confirms the **architecture**, not enterprise scale — and in production the diagnosis is only as good as your real lineage source.

## Setup

One install cell. `openai` / `anthropic` are only exercised in live mode — MOCK mode needs neither at runtime.

In [ ]:
%pip install -q pydantic anthropic openai python-dotenv pandas matplotlib

## Pick a mode

MOCK is the default: the full pipeline runs instantly with canned (but Pydantic-validated) agent responses. To go live, put `OPENAI_API_KEY` or `ANTHROPIC_API_KEY` (and optionally `MODEL`) in a `.env` file next to this notebook's repo root. Anthropic wins if both are set. There is no silent fallback: live mode errors surface as errors.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

if os.getenv("ANTHROPIC_API_KEY"):
    PROVIDER, MODEL = "anthropic", os.getenv("MODEL", "claude-sonnet-4-6")
elif os.getenv("OPENAI_API_KEY"):
    PROVIDER, MODEL = "openai", os.getenv("MODEL", "gpt-4o-mini")
else:
    PROVIDER, MODEL = "mock", None

print("MOCK mode — no LLM calls, canned responses" if PROVIDER == "mock"
      else f"LIVE mode — {PROVIDER} -> {MODEL}")

---
## Part 1 — The shop (with a planted bug)

Five products, ~40 orders a day, 15 days. `seed_shop` plants one of two bugs on the **last day**:

- `price_glitch` — someone fat-fingers the Coffee Mug's price to **$120 instead of $12** (a `manual_price_update` change event at 08:00). The mug is the best-seller (~40% of orders), so average order value jumps ~3x.
- `volume_drop` — the `order_ingest` pipeline fails at 06:30 and ~80% of the day's orders never arrive.

Lineage is the backbone: `products → orders → daily_sales → {sales_dashboard, finance_report, demand_forecast}`. Here it's a dict; in production it's your lineage tool behind an MCP server — same contract either way.

In [ ]:
import sqlite3, random
from datetime import date, timedelta

PRODUCTS = [
    # (sku, name, price, share of order volume)
    ("MUG-01", "Coffee Mug",   12.0, 0.40),
    ("TSH-02", "T-Shirt",      25.0, 0.20),
    ("NBK-03", "Notebook",      8.0, 0.15),
    ("BTL-04", "Water Bottle", 18.0, 0.15),
    ("HDY-05", "Hoodie",       40.0, 0.10),
]

def seed_shop(n_days=15, bug_type="price_glitch", magnitude=10.0, orders_per_day=40, seed=42):
    """In-memory shop warehouse with a bug planted on the last day.
    bug_type: "price_glitch" (magnitude = price multiplier) | "volume_drop" (magnitude = % of orders lost)
    """
    rng = random.Random(seed)
    base = date(2026, 7, 1)
    days = [base + timedelta(days=i) for i in range(n_days)]

    con = sqlite3.connect(":memory:")
    c = con.cursor()
    c.execute("CREATE TABLE products(sku TEXT, name TEXT, price REAL)")
    c.execute("CREATE TABLE orders(id INTEGER, date TEXT, sku TEXT, qty INTEGER, unit_price REAL, total REAL)")
    c.execute("CREATE TABLE daily_sales(date TEXT, total_sales REAL)")

    for sku, name, price, _ in PRODUCTS:
        c.execute("INSERT INTO products VALUES(?,?,?)", (sku, name, price))

    skus    = [p[0] for p in PRODUCTS]
    prices  = {p[0]: p[2] for p in PRODUCTS}
    weights = [p[3] for p in PRODUCTS]

    oid = 0
    for i, d in enumerate(days):
        is_bug_day = (i == n_days - 1)
        day_prices = dict(prices)
        if is_bug_day and bug_type == "price_glitch":
            day_prices["MUG-01"] = round(prices["MUG-01"] * magnitude, 2)
        n_orders = orders_per_day
        if is_bug_day and bug_type == "volume_drop":
            n_orders = max(1, int(orders_per_day * (1 - magnitude / 100)))
        for _ in range(n_orders):
            oid += 1
            sku = rng.choices(skus, weights=weights)[0]
            qty = rng.randint(1, 3)
            c.execute("INSERT INTO orders VALUES(?,?,?,?,?,?)",
                      (oid, d.isoformat(), sku, qty, day_prices[sku],
                       round(qty * day_prices[sku], 2)))

    for d in days:
        tot = c.execute("SELECT SUM(total) FROM orders WHERE date=?", (d.isoformat(),)).fetchone()[0] or 0.0
        c.execute("INSERT INTO daily_sales VALUES(?,?)", (d.isoformat(), round(tot, 2)))
    con.commit()

    LINEAGE = {
        "products":    {"upstream": [],                           "downstream": ["orders"]},
        "orders":      {"upstream": ["products", "order_ingest"], "downstream": ["daily_sales"]},
        "daily_sales": {"upstream": ["orders"],
                        "downstream": ["sales_dashboard", "finance_report", "demand_forecast"]},
    }
    CATALOG = {
        "orders":          {"criticality": "medium", "sensitivity": "financial"},
        "daily_sales":     {"criticality": "high",   "sensitivity": "financial"},
        "sales_dashboard": {"type": "dashboard",     "criticality": "high"},
        "finance_report":  {"type": "report",        "criticality": "regulatory"},
        "demand_forecast": {"type": "ml_feature",    "criticality": "medium"},
    }
    if bug_type == "price_glitch":
        CHANGES = [{"asset": "products",     "event": "manual_price_update", "at": f"{days[-1].isoformat()}T08:00Z"}]
    else:
        CHANGES = [{"asset": "order_ingest", "event": "pipeline_failure",    "at": f"{days[-1].isoformat()}T06:30Z"}]

    return con, c, days, LINEAGE, CATALOG, CHANGES

con, c, days, LINEAGE, CATALOG, CHANGES = seed_shop()
n_orders = c.execute("SELECT COUNT(*) FROM orders").fetchone()[0]
top_price = c.execute("SELECT MAX(unit_price) FROM orders WHERE date=?", (days[-1].isoformat(),)).fetchone()[0]
print(f"Seeded {len(days)} days, {n_orders} orders. Top unit price on the last day: ${top_price} (Coffee Mug normally $12)")

### See the anomaly before any AI runs

Detection shouldn't be the impressive part. Your eye finds the problem instantly — the detectors just automate that glance.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def plot_daily_sales(con, title="Daily sales (USD)"):
    df = pd.read_sql_query("SELECT date, total_sales FROM daily_sales ORDER BY date", con)
    fig, ax = plt.subplots(figsize=(9, 3.2))
    ax.plot(range(len(df)), df["total_sales"], color="#2f6db3", linewidth=2,
            marker="o", markersize=4)
    last_i, last_v = len(df) - 1, df["total_sales"].iloc[-1]
    ax.annotate(f"${last_v:,.0f} ?!", (last_i, last_v), textcoords="offset points",
                xytext=(-66, -4), fontweight="bold", color="#b0413e")
    ax.set_xticks(range(len(df)))
    ax.set_xticklabels([d[5:] for d in df["date"]], rotation=45, fontsize=8)
    ax.set_title(title, loc="left", fontweight="bold")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.show()
    return df

_ = plot_daily_sales(con, "Daily sales (USD) — spot the last day")